## Importing Libraries

In [197]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Loading Dataset

In [149]:
df = pd.read_csv('../data/processed/rainfall_cleaned.csv')

In [150]:
df.head()

,SUBDIVISION,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANNUAL,Jan-Feb,Mar-May,Jun-Sep,Oct-Dec
0,ANDAMAN & NICOBAR ISLANDS,1901,49.2,87.1,29.2,2.3,528.8,517.5,365.1,481.1,332.6,388.5,558.2,33.6,3373.2,136.3,560.3,1696.3,980.3
1,ANDAMAN & NICOBAR ISLANDS,1902,0.0,159.8,12.2,0.0,446.1,537.1,228.9,753.7,666.2,197.2,359.0,160.5,3520.7,159.8,458.3,2185.9,716.7
2,ANDAMAN & NICOBAR ISLANDS,1903,12.7,144.0,0.0,1.0,235.1,479.9,728.4,326.7,339.0,181.2,284.4,225.0,2957.4,156.7,236.1,1874.0,690.6
3,ANDAMAN & NICOBAR ISLANDS,1904,9.4,14.7,0.0,202.4,304.5,495.1,502.0,160.1,820.4,222.2,308.7,40.1,3079.6,24.1,506.9,1977.6,571.0
4,ANDAMAN & NICOBAR ISLANDS,1905,1.3,0.0,3.3,26.9,279.5,628.7,368.7,330.5,297.0,260.7,25.4,344.7,2566.7,1.3,309.7,1624.9,630.8


In [151]:
df.shape

(4116, 19)

In [152]:
df[['SUBDIVISION', 'YEAR', 'ANNUAL']].head()

,SUBDIVISION,YEAR,ANNUAL
0,ANDAMAN & NICOBAR ISLANDS,1901,3373.2
1,ANDAMAN & NICOBAR ISLANDS,1902,3520.7
2,ANDAMAN & NICOBAR ISLANDS,1903,2957.4
3,ANDAMAN & NICOBAR ISLANDS,1904,3079.6
4,ANDAMAN & NICOBAR ISLANDS,1905,2566.7


In [153]:
df[['SUBDIVISION', 'YEAR', 'ANNUAL']].tail()

,SUBDIVISION,YEAR,ANNUAL
4111,LAKSHADWEEP,2011,1533.7
4112,LAKSHADWEEP,2012,1405.5
4113,LAKSHADWEEP,2013,1426.3
4114,LAKSHADWEEP,2014,1395.0
4115,LAKSHADWEEP,2015,1642.9


In [154]:
df = df.sort_values(
    ['SUBDIVISION', 'YEAR']
).reset_index(drop=True)

In [155]:
df['year_diff'] = (
    df.groupby('SUBDIVISION')['YEAR']
      .diff()
)

In [156]:
gaps = df[df['year_diff'] > 1]
gaps[['SUBDIVISION', 'YEAR', 'year_diff']].head()

,SUBDIVISION,YEAR,year_diff
8,ANDAMAN & NICOBAR ISLANDS,1910,2.0
41,ANDAMAN & NICOBAR ISLANDS,1946,4.0
43,ANDAMAN & NICOBAR ISLANDS,1949,2.0
148,ARUNACHAL PRADESH,1957,4.0
2064,LAKSHADWEEP,1919,2.0


In [157]:
print("Number of year gaps:", len(gaps))

Number of year gaps: 5


In [158]:
df = df.drop(columns=['year_diff'])

#### Create Lag1

In [159]:
previous_year = df[
    ['SUBDIVISION', 'YEAR', 'ANNUAL']
].copy()

previous_year['YEAR'] = previous_year['YEAR'] + 1

previous_year = previous_year.rename(
    columns={'ANNUAL': 'annual_lag1'}
)

In [160]:
df = df.merge(
    previous_year[
        ['SUBDIVISION', 'YEAR', 'annual_lag1']
    ],
    on=['SUBDIVISION', 'YEAR'],
    how='left'
)

#### Create Lag2

In [161]:
lag2 = df[
    ['SUBDIVISION', 'YEAR', 'ANNUAL']
].copy()

lag2['YEAR'] = lag2['YEAR'] + 2

lag2 = lag2.rename(
    columns={'ANNUAL': 'annual_lag2'}
)

In [162]:
df = df.merge(
    lag2[
        ['SUBDIVISION', 'YEAR', 'annual_lag2']
    ],
    on=['SUBDIVISION', 'YEAR'],
    how='left'
)

#### Create Lag3

In [163]:
lag3 = df[
    ['SUBDIVISION', 'YEAR', 'ANNUAL']
].copy()

lag3['YEAR'] = lag3['YEAR'] + 3

lag3 = lag3.rename(
    columns={'ANNUAL': 'annual_lag3'}
)

In [164]:
df = df.merge(
    lag3[
        ['SUBDIVISION', 'YEAR', 'annual_lag3']
    ],
    on=['SUBDIVISION', 'YEAR'],
    how='left'
)

#### Create Rolling Mean

In [165]:
df['rolling_mean_3'] = df[
    ['annual_lag1', 'annual_lag2', 'annual_lag3']
].mean(axis=1, skipna=False)

#### Five_Year Rolling Mean

In [166]:
df['annual_lag4'] = [
    annual_lookup.get((subdivision, year - 4), np.nan)
    for subdivision, year in zip(
        df['SUBDIVISION'],
        df['YEAR']
    )
]

df['annual_lag5'] = [
    annual_lookup.get((subdivision, year - 5), np.nan)
    for subdivision, year in zip(
        df['SUBDIVISION'],
        df['YEAR']
    )
]

df['rolling_mean_5'] = df[
    [
        'annual_lag1',
        'annual_lag2',
        'annual_lag3',
        'annual_lag4',
        'annual_lag5'
    ]
].mean(axis=1, skipna=False)

#### Create the Target

In [167]:
df['target'] = [
    annual_lookup.get((subdivision, year + 1), np.nan)
    for subdivision, year in zip(
        df['SUBDIVISION'],
        df['YEAR']
    )
]

In [168]:
df[
    (df['SUBDIVISION'] == 'ANDAMAN & NICOBAR ISLANDS') &
    (df['YEAR'].between(1907, 1911))
][
    ['YEAR', 'ANNUAL', 'target']
]

,YEAR,ANNUAL,target
6,1907,3347.9,3576.4
7,1908,3576.4,NaN
8,1910,2899.4,2687.2
9,1911,2687.2,2960.5


In [169]:
df.head()

,SUBDIVISION,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,...,Jun-Sep,Oct-Dec,annual_lag1,annual_lag2,annual_lag3,rolling_mean_3,annual_lag4,annual_lag5,rolling_mean_5,target
0,ANDAMAN & NICOBAR ISLANDS,1901,49.2,87.1,29.2,2.3,528.8,517.5,365.1,481.1,...,1696.3,980.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3520.7
1,ANDAMAN & NICOBAR ISLANDS,1902,0.0,159.8,12.2,0.0,446.1,537.1,228.9,753.7,...,2185.9,716.7,3373.2,NaN,NaN,NaN,NaN,NaN,NaN,2957.4
2,ANDAMAN & NICOBAR ISLANDS,1903,12.7,144.0,0.0,1.0,235.1,479.9,728.4,326.7,...,1874.0,690.6,3520.7,3373.2,NaN,NaN,NaN,NaN,NaN,3079.6
3,ANDAMAN & NICOBAR ISLANDS,1904,9.4,14.7,0.0,202.4,304.5,495.1,502.0,160.1,...,1977.6,571.0,2957.4,3520.7,3373.2,3283.766667,NaN,NaN,NaN,2566.7
4,ANDAMAN & NICOBAR ISLANDS,1905,1.3,0.0,3.3,26.9,279.5,628.7,368.7,330.5,...,1624.9,630.8,3079.6,2957.4,3520.7,3185.900000,3373.2,NaN,NaN,2534.4


In [170]:
df.isnull().sum()

SUBDIVISION         0
YEAR                0
JAN                 0
FEB                 0
MAR                 0
APR                 0
MAY                 0
JUN                 0
JUL                 0
AUG                 0
SEP                 0
OCT                 0
NOV                 0
DEC                 0
ANNUAL              0
Jan-Feb             0
Mar-May             0
Jun-Sep             0
Oct-Dec             0
annual_lag1        41
annual_lag2        79
annual_lag3       116
rolling_mean_3    122
annual_lag4       152
annual_lag5       188
rolling_mean_5    202
target             41
dtype: int64

#### Validation

In [171]:
# Create a lookup table for annual rainfall
annual_lookup = df.set_index(
    ['SUBDIVISION', 'YEAR']
)['ANNUAL']

In [172]:
# Create three lags
df['annual_lag1'] = [
    annual_lookup.get((subdivision, year - 1), np.nan)
    for subdivision, year in zip(
        df['SUBDIVISION'],
        df['YEAR']
    )
]

df['annual_lag2'] = [
    annual_lookup.get((subdivision, year - 2), np.nan)
    for subdivision, year in zip(
        df['SUBDIVISION'],
        df['YEAR']
    )
]

df['annual_lag3'] = [
    annual_lookup.get((subdivision, year - 3), np.nan)
    for subdivision, year in zip(
        df['SUBDIVISION'],
        df['YEAR']
    )
]

In [173]:
# Check of the gap
df[
    (df['SUBDIVISION'] == 'ANDAMAN & NICOBAR ISLANDS') &
    (df['YEAR'].between(1907, 1911))
][
    [
        'SUBDIVISION',
        'YEAR',
        'ANNUAL',
        'annual_lag1',
        'annual_lag2',
        'annual_lag3'
    ]
]

,SUBDIVISION,YEAR,ANNUAL,annual_lag1,annual_lag2,annual_lag3
6,ANDAMAN & NICOBAR ISLANDS,1907,3347.9,2534.4,2566.7,3079.6
7,ANDAMAN & NICOBAR ISLANDS,1908,3576.4,3347.9,2534.4,2566.7
8,ANDAMAN & NICOBAR ISLANDS,1910,2899.4,NaN,3576.4,3347.9
9,ANDAMAN & NICOBAR ISLANDS,1911,2687.2,2899.4,NaN,3576.4


In [174]:
df[
    (df['SUBDIVISION'] == 'ANDAMAN & NICOBAR ISLANDS') &
    (df['YEAR'].between(1907, 1911))
][
    [
        'YEAR',
        'ANNUAL',
        'annual_lag1',
        'annual_lag2',
        'annual_lag3',
        'rolling_mean_3'
    ]
]

,YEAR,ANNUAL,annual_lag1,annual_lag2,annual_lag3,rolling_mean_3
6,1907,3347.9,2534.4,2566.7,3079.6,2726.900000
7,1908,3576.4,3347.9,2534.4,2566.7,2816.333333
8,1910,2899.4,NaN,3576.4,3347.9,NaN
9,1911,2687.2,2899.4,NaN,3576.4,NaN


In [175]:
df.to_csv(
    '../data/processed/rainfall_features.csv',
    index=False
)

## Feature Engineering — Summary

### Create Seasonal Lookup

In [176]:
seasonal_lookup = {
    'Jan-Feb': df.set_index(
        ['SUBDIVISION', 'YEAR']
    )['Jan-Feb'],

    'Mar-May': df.set_index(
        ['SUBDIVISION', 'YEAR']
    )['Mar-May'],

    'Jun-Sep': df.set_index(
        ['SUBDIVISION', 'YEAR']
    )['Jun-Sep'],

    'Oct-Dec': df.set_index(
        ['SUBDIVISION', 'YEAR']
    )['Oct-Dec']
}

### Create Seasonal Lag Features

In [177]:
df['season_janfeb_lag1'] = [
    seasonal_lookup['Jan-Feb'].get(
        (subdivision, year - 1),
        np.nan
    )
    for subdivision, year in zip(
        df['SUBDIVISION'],
        df['YEAR']
    )
]

df['season_marmay_lag1'] = [
    seasonal_lookup['Mar-May'].get(
        (subdivision, year - 1),
        np.nan
    )
    for subdivision, year in zip(
        df['SUBDIVISION'],
        df['YEAR']
    )
]

df['season_jjason_lag1'] = [
    seasonal_lookup['Jun-Sep'].get(
        (subdivision, year - 1),
        np.nan
    )
    for subdivision, year in zip(
        df['SUBDIVISION'],
        df['YEAR']
    )
]

df['season_octdec_lag1'] = [
    seasonal_lookup['Oct-Dec'].get(
        (subdivision, year - 1),
        np.nan
    )
    for subdivision, year in zip(
        df['SUBDIVISION'],
        df['YEAR']
    )
]

### Check the Features


In [178]:
seasonal_features = [
    'season_janfeb_lag1',
    'season_marmay_lag1',
    'season_jjason_lag1',
    'season_octdec_lag1'
]

print(
    df[
        [
            'SUBDIVISION',
            'YEAR',
            'ANNUAL',
            'season_janfeb_lag1',
            'season_marmay_lag1',
            'season_jjason_lag1',
            'season_octdec_lag1'
        ]
    ].head(10)
)

                 SUBDIVISION  YEAR  ANNUAL  season_janfeb_lag1  \
0  ANDAMAN & NICOBAR ISLANDS  1901  3373.2                 NaN   
1  ANDAMAN & NICOBAR ISLANDS  1902  3520.7               136.3   
2  ANDAMAN & NICOBAR ISLANDS  1903  2957.4               159.8   
3  ANDAMAN & NICOBAR ISLANDS  1904  3079.6               156.7   
4  ANDAMAN & NICOBAR ISLANDS  1905  2566.7                24.1   
5  ANDAMAN & NICOBAR ISLANDS  1906  2534.4                 1.3   
6  ANDAMAN & NICOBAR ISLANDS  1907  3347.9                36.6   
7  ANDAMAN & NICOBAR ISLANDS  1908  3576.4               110.7   
8  ANDAMAN & NICOBAR ISLANDS  1910  2899.4                 NaN   
9  ANDAMAN & NICOBAR ISLANDS  1911  2687.2                49.3   

   season_marmay_lag1  season_jjason_lag1  season_octdec_lag1  
0                 NaN                 NaN                 NaN  
1               560.3              1696.3               980.3  
2               458.3              2185.9               716.7  
3               2

In [179]:
#Validate missing values
print(
    df[seasonal_features].isna().sum()
)

season_janfeb_lag1    41
season_marmay_lag1    41
season_jjason_lag1    41
season_octdec_lag1    41
dtype: int64


In [180]:
#Verify the important missing-year case
print(
    df[
        (df['SUBDIVISION'] == 'ANDAMAN & NICOBAR ISLANDS') &
        (df['YEAR'].between(1907, 1911))
    ][
        [
            'YEAR',
            'Jun-Sep',
            'season_jjason_lag1'
        ]
    ]
)

   YEAR  Jun-Sep  season_jjason_lag1
6  1907   1327.1              1465.8
7  1908   2303.7              1327.1
8  1910   1701.0                 NaN
9  1911   1553.6              1701.0


### Train Test Split

In [181]:
train_df = df[df['YEAR'] <= 2000].copy()

validation_df = df[
    (df['YEAR'] > 2000) &
    (df['YEAR'] <= 2008)
].copy()

test_df = df[
    (df['YEAR'] > 2008) &
    (df['YEAR'] <= 2014)
].copy()

In [182]:
feature_cols_exp5 = [
    'annual_lag1',
    'annual_lag2',
    'annual_lag3',
    'annual_lag4',
    'annual_lag5',
    'rolling_mean_3',
    'rolling_mean_5',
    'season_janfeb_lag1',
    'season_marmay_lag1',
    'season_jjason_lag1',
    'season_octdec_lag1'
]

In [183]:
X_train_exp5 = train_df[feature_cols_exp5]
y_train_exp5 = train_df['target']

X_validation_exp5 = validation_df[feature_cols_exp5]
y_validation_exp5 = validation_df['target']

X_test_exp5 = test_df[feature_cols_exp5]
y_test_exp5 = test_df['target']

In [184]:
print("Train missing:", X_train_exp5.isna().sum().sum())
print("Validation missing:", X_validation_exp5.isna().sum().sum())
print("Test missing:", X_test_exp5.isna().sum().sum())

Train missing: 1064
Validation missing: 0
Test missing: 0


### Remove Incomplete Training Rows

In [185]:
train_complete = (
    train_df[feature_cols_exp5 + ['target']]
    .dropna()
    .copy()
)

print("Original training rows:", len(train_df))
print("Complete training rows:", len(train_complete))
print("Rows removed:", len(train_df) - len(train_complete))

Original training rows: 3576
Complete training rows: 3370
Rows removed: 206


In [186]:
X_train_exp5 = train_complete[feature_cols_exp5]
y_train_exp5 = train_complete['target']

In [187]:
X_validation_exp5 = validation_df[feature_cols_exp5]
y_validation_exp5 = validation_df['target']

X_test_exp5 = test_df[feature_cols_exp5]
y_test_exp5 = test_df['target']

In [188]:
print("X_train shape:", X_train_exp5.shape)
print("y_train shape:", y_train_exp5.shape)

print("X_validation shape:", X_validation_exp5.shape)
print("y_validation shape:", y_validation_exp5.shape)

print("X_test shape:", X_test_exp5.shape)
print("y_test shape:", y_test_exp5.shape)

print("\nMissing values:")
print("Train:", X_train_exp5.isna().sum().sum())
print("Validation:", X_validation_exp5.isna().sum().sum())
print("Test:", X_test_exp5.isna().sum().sum())

X_train shape: (3370, 11)
y_train shape: (3370,)
X_validation shape: (288, 11)
y_validation shape: (288,)
X_test shape: (216, 11)
y_test shape: (216,)

Missing values:
Train: 0
Validation: 0
Test: 0


In [194]:
linear_model_exp5 = LinearRegression()

linear_model_exp5.fit(
    X_train_exp5,
    y_train_exp5
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [195]:
validation_pred_exp5 = linear_model_exp5.predict(
    X_validation_exp5
)

test_pred_exp5 = linear_model_exp5.predict(
    X_test_exp5
)

In [198]:
validation_mae_exp5 = mean_absolute_error(
    y_validation_exp5,
    validation_pred_exp5
)

validation_rmse_exp5 = np.sqrt(
    mean_squared_error(
        y_validation_exp5,
        validation_pred_exp5
    )
)

validation_r2_exp5 = r2_score(
    y_validation_exp5,
    validation_pred_exp5
)

test_mae_exp5 = mean_absolute_error(
    y_test_exp5,
    test_pred_exp5
)

test_rmse_exp5 = np.sqrt(
    mean_squared_error(
        y_test_exp5,
        test_pred_exp5
    )
)

test_r2_exp5 = r2_score(
    y_test_exp5,
    test_pred_exp5
)

print("Experiment 5 - Validation")
print("MAE:", validation_mae_exp5)
print("RMSE:", validation_rmse_exp5)
print("R²:", validation_r2_exp5)

print("\nExperiment 5 - Test")
print("MAE:", test_mae_exp5)
print("RMSE:", test_rmse_exp5)
print("R²:", test_r2_exp5)

Experiment 5 - Validation
MAE: 223.79171903715948
RMSE: 290.6023834150841
R²: 0.8781226800726833

Experiment 5 - Test
MAE: 212.3071036630603
RMSE: 290.68626069252184
R²: 0.889811345586492
